In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/Users/HP/Documents/second-publication/meetingTranscript")
RUN_CSV = BASE_DIR / "semantic-fidelity" / "bertscore_semantic_fidelity.csv"
AUTHOR_CSV = BASE_DIR / "semantic-fidelity" / "bertscore_author_versions.csv"
OUTPUT_TEX = BASE_DIR / "semantic-fidelity" / "text_cleaning_fidelity_table.tex"

# Section selections from the 3 initial runs
RETAINED_RUNS = {
    "elicitation_process": "cleaned_contexts_run02_20260629_104239.json",
    "project_description": "cleaned_contexts_run03_20260629_104252.json",
    "demonstrator_description": "cleaned_contexts_run01_20260629_104226.json",
}

SECTION_ORDER = [
    "elicitation_process",
    "project_description",
    "demonstrator_description",
]

def simplify_name(name: str) -> str:
    name = Path(name).name
    name = name.replace(".json", "").replace(".jsonl", "")
    return name

# ---------- Load run-selection rows ----------
run_df = pd.read_csv(RUN_CSV)
run_df = run_df[run_df["section"] != "__overall__"].copy()

run_df["status"] = run_df.apply(
    lambda row: (
        "Candidate retained"
        if RETAINED_RUNS.get(row["section"]) == row["cleaned_file"]
        else "Candidate not retained"
    ),
    axis=1,
)

run_df = run_df.rename(columns={"cleaned_file": "edited_file"})
run_df["raw_file"] = "transcription"

run_df = run_df[
    ["raw_file", "edited_file", "section", "precision", "recall", "f1", "status"]
].copy()

# ---------- Load author-version rows ----------
author_df = pd.read_csv(AUTHOR_CSV)
author_df = author_df[author_df["section"] != "__overall__"].copy()

author_df["status"] = "Final revised version"

author_df = author_df[
    ["raw_file", "edited_file", "section", "precision", "recall", "f1", "status"]
].copy()

# ---------- Combine ----------
final_df = pd.concat([run_df, author_df], ignore_index=True)

final_df["raw_file"] = final_df["raw_file"].map(simplify_name)
final_df["edited_file"] = final_df["edited_file"].map(simplify_name)

final_df["section"] = pd.Categorical(
    final_df["section"],
    categories=SECTION_ORDER,
    ordered=True
)

final_df = final_df.sort_values(["section", "status", "edited_file"]).reset_index(drop=True)

# Round scores for presentation
for col in ["precision", "recall", "f1"]:
    final_df[col] = final_df[col].round(3)

# Optional: nicer section labels for the table
section_labels = {
    "elicitation_process": "Elicitation process",
    "project_description": "Project description",
    "demonstrator_description": "Demonstrator description",
}
final_df["section"] = final_df["section"].map(section_labels)

# ---------- Export LaTeX ----------
latex_table = final_df.to_latex(
    index=False,
    escape=False,
    caption="Section-level semantic fidelity assessment for the initial run-selection and the final author revision.",
    label="tab:text_cleaning_semantic_fidelity",
    column_format="ll l c c c l"
)

OUTPUT_TEX.write_text(latex_table, encoding="utf-8")

print(final_df)
print(f"\nLaTeX table written to:\n{OUTPUT_TEX}")

             raw_file                             edited_file  \
0       transcription  cleaned_contexts_run01_20260629_104226   
1       transcription  cleaned_contexts_run03_20260629_104252   
2       transcription  cleaned_contexts_run02_20260629_104239   
3   author-review-raw                   author-review-editedl   
4       transcription  cleaned_contexts_run01_20260629_104226   
5       transcription  cleaned_contexts_run02_20260629_104239   
6       transcription  cleaned_contexts_run03_20260629_104252   
7   author-review-raw                   author-review-editedl   
8       transcription  cleaned_contexts_run02_20260629_104239   
9       transcription  cleaned_contexts_run03_20260629_104252   
10      transcription  cleaned_contexts_run01_20260629_104226   
11  author-review-raw                   author-review-editedl   

                     section  precision  recall     f1                  status  
0        Elicitation process      0.820   0.823  0.822  Candidate not ret